# Baselines & Collaborative Filtering

## 1. Data Preparation

In [48]:
from recsys_common import *

configure_notebook()

project_dir = Path.cwd()
shared_dir = project_dir / "artifacts" / "shared"

required_shared = [
    shared_dir / "df_model.csv",
    shared_dir / "train_df.csv",
    shared_dir / "test_df.csv",
]
missing_shared = [str(p) for p in required_shared if not p.exists()]
if missing_shared:
    raise FileNotFoundError(
        "Shared artifacts are required. Run 0_Data_Exploration.ipynb first. Missing:\\n"
        + "\\n".join(missing_shared)
    )

print("Loading prepared data from artifacts/shared/")
df_model = pd.read_csv(shared_dir / "df_model.csv")
train_df = pd.read_csv(shared_dir / "train_df.csv")
test_df = pd.read_csv(shared_dir / "test_df.csv")

for frame in (df_model, train_df, test_df):
    if "review_time" in frame.columns:
        frame["review_time"] = pd.to_datetime(frame["review_time"], errors="coerce")

print(f"Shared df_model: {df_model.shape}")
print(f"Shared train_df: {train_df.shape}")
print(f"Shared test_df: {test_df.shape}")


Loading prepared data from artifacts/shared/
Shared df_model: (268743, 15)
Shared train_df: (155052, 15)
Shared test_df: (113691, 15)


In [49]:
print("Model-specific imports are provided by recsys_common.py")

Model-specific imports are provided by recsys_common.py


In [50]:
relevance_threshold = 4
df_model["is_relevant"] = (df_model["Score"] >= relevance_threshold).astype(int)

print("Relevant interaction rate:", round(df_model["is_relevant"].mean(), 4))

Relevant interaction rate: 0.7814


In [51]:
print("Using shared split loaded from artifacts/shared/")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train users:", train_df["UserId"].nunique())
print("Test users:", test_df["UserId"].nunique())
print("Train products:", train_df["ProductId"].nunique())
print("Test products:", test_df["ProductId"].nunique())

Using shared split loaded from artifacts/shared/
Train shape: (155052, 15)
Test shape: (113691, 15)
Train users: 17678
Test users: 79204
Train products: 14279
Test products: 34110


## 2. Evaluation Helper Functions

In [52]:
K = 10

print(f"Ranking cutoff K: {K}")
print(f"Relevance threshold: {relevance_threshold}")

Ranking cutoff K: 10
Relevance threshold: 4


In [53]:
print("Shared evaluation helpers loaded from recsys_common.py")

Shared evaluation helpers loaded from recsys_common.py


Evaluation note: ranking metrics here are computed on each user's held-out test interactions (not on full-catalog candidate ranking). This is acceptable for lightweight offline comparison, but values can be optimistic versus a full Top-N candidate evaluation setting.


################

## **6. Baseline Recommenders**

The random baseline recommends unseen products at random. It does not use ratings, popularity, or similarity, so it serves as a weak lower-bound benchmark.

### 6.1 Prepare data for surpise

To keep model training and evaluation consistent across baselines and collaborative filtering methods, we use the `surprise` library. It is designed for explicit-feedback recommendation tasks and provides a clean framework for rating prediction.

In [54]:
reader = Reader(rating_scale=(1, 5))

# Use the SAME split as train_df/test_df to avoid mismatch and leakage
train_data = Dataset.load_from_df(train_df[["UserId", "ProductId", "Score"]], reader)
trainset = train_data.build_full_trainset()

# Surprise testset format: list of (uid, iid, true_rating)
testset = list(test_df[["UserId", "ProductId", "Score"]].itertuples(index=False, name=None))

print("Surprise trainset size:", trainset.n_ratings)
print("Surprise testset size:", len(testset))

Surprise trainset size: 155052
Surprise testset size: 113691


In [ ]:
TOP_N = 10
MAX_CATALOG_USERS = 300   # set to None if runtime is fine and you want all eligible users

# Use TRAIN catalog for model-compatible candidate generation.
all_items = sorted(train_df["ProductId"].dropna().unique())

train_seen_items = train_df.groupby("UserId")["ProductId"].apply(set).to_dict()

eligible_catalog_users = [
    user_id for user_id in sorted(test_df["UserId"].unique())
    if len(set(all_items) - train_seen_items.get(user_id, set())) > 0
]

if MAX_CATALOG_USERS is None:
    catalog_users = eligible_catalog_users
else:
    catalog_users = eligible_catalog_users[:MAX_CATALOG_USERS]

item_codes, item_labels = pd.factorize(train_df["ProductId"])
user_codes, user_labels = pd.factorize(train_df["UserId"])

item_user_matrix = csr_matrix(
    (train_df["Score"].astype(float), (item_codes, user_codes)),
    shape=(len(item_labels), len(user_labels))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_labels)}

print("Catalog users evaluated:", len(catalog_users))
print("Catalog size:", len(all_items))
print("TOP_N:", TOP_N)

Catalog users evaluated: 300
Catalog size: 14279
TOP_N: 10


### 6.2 Random recommender

In [56]:
np.random.seed(42)

random_model = NormalPredictor()
random_model.fit(trainset)

random_predictions = random_model.test(testset)

random_rmse = accuracy.rmse(random_predictions, verbose=False)
random_mae = accuracy.mae(random_predictions, verbose=False)

print("Random Baseline")
print(f"RMSE: {random_rmse:.4f}")
print(f"MAE:  {random_mae:.4f}")

Random Baseline
RMSE: 1.6304
MAE:  1.2179


#### 6.2.1 Ranking Metrics

In [10]:
random_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in random_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

random_eval_df.head()

,UserId,ProductId,Score,pred
0,A1L01D2BD3RKVO,B000EVG8J2,5,4.805578
1,A3U62RE5XZDP0G,B0000BXJIS,5,4.019986
2,AOXC0JQQZGGB6,B008FHUFAU,3,4.992363
3,A3PWPNZVMNX3PA,B006BXV14E,2,5.000000
4,A1XNZ7PCE45KK7,B007I7Z3Z0,5,3.901352


In [57]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
random_topn = build_topn_recommendations(
    lambda user_id, item_id: random_model.predict(user_id, item_id).est,
    top_n=TOP_N,
 )

random_rank = ranking_metrics_from_topn(
    random_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

random_precision = random_rank["precision"]
random_recall = random_rank["recall"]
random_map = random_rank["map"]
random_ndcg = random_rank["ndcg"]

print("Random Baseline Ranking Metrics")
print(f"Precision@{K}: {random_precision:.8f}")
print(f"Recall@{K}: {random_recall:.8f}")
print(f"MAP@{K}: {random_map:.8f}")
print(f"NDCG@{K}: {random_ndcg:.8f}")

Random Baseline Ranking Metrics
Precision@10: 0.00000000
Recall@10: 0.00000000
MAP@10: 0.00000000
NDCG@10: 0.00000000


#### 6.2.2 Catalog Metrics

In [12]:
if "random_topn" not in globals():
    random_topn = build_topn_recommendations(
        lambda user_id, item_id: random_model.predict(user_id, item_id).est,
        top_n=TOP_N,
    )

print("Random full-candidate Top-N lists generated:", len(random_topn))

Random full-candidate Top-N lists generated: 300


In [13]:
random_coverage = coverage_at_n(random_topn, all_items)
random_personalization = personalization_at_n(random_topn, all_items)
random_ils = intra_list_similarity_at_n(random_topn, item_similarity, item_to_idx)

print("Random Baseline Behavioral/Catalog Metrics")
print(f"Coverage: {random_coverage:.4f}")
print(f"Personalization: {random_personalization:.4f}")
print(f"Intra-list similarity: {random_ils:.4f}")

Random Baseline Behavioral/Catalog Metrics
Coverage: 0.0047
Personalization: 0.7813
Intra-list similarity: 0.0102


In [14]:
display(qualitative_examples(random_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,B00004RBDZ,5.0
1,#oc-R115TNMSPFT9I7,2,B00004RYGX,5.0
2,#oc-R115TNMSPFT9I7,3,B00004S1C5,5.0
3,#oc-R115TNMSPFT9I7,4,B00004S1C6,5.0
4,#oc-R115TNMSPFT9I7,5,B00005344V,5.0
5,#oc-R115TNMSPFT9I7,6,B00005IX96,5.0
6,#oc-R115TNMSPFT9I7,7,B00005IX98,5.0
7,#oc-R115TNMSPFT9I7,8,B00006LL38,5.0
8,#oc-R115TNMSPFT9I7,9,B000084DWM,5.0
9,#oc-R115TNMSPFT9I7,10,B000084E76,5.0


### 6.3 Baseline bias model

`BaselineOnly` predicts ratings using user and item bias terms. It is still simple, but more informed than a random predictor because it captures tendencies such as some users rating higher on average and some products receiving higher ratings overall.

In [15]:
baseline_model = BaselineOnly()
baseline_model.fit(trainset)

baseline_predictions = baseline_model.test(testset)

baseline_rmse = accuracy.rmse(baseline_predictions, verbose=False)
baseline_mae = accuracy.mae(baseline_predictions, verbose=False)


print("Baseline Bias Model")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"MAE:  {baseline_mae:.4f}")

Estimating biases using als...
Baseline Bias Model
RMSE: 1.2215
MAE:  0.9464


#### 6.3.1 Ranking Metrics

In [16]:
bias_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in baseline_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

bias_eval_df.head()

,UserId,ProductId,Score,pred
0,A1L01D2BD3RKVO,B000EVG8J2,5,4.501326
1,A3U62RE5XZDP0G,B0000BXJIS,5,4.191046
2,AOXC0JQQZGGB6,B008FHUFAU,3,4.015292
3,A3PWPNZVMNX3PA,B006BXV14E,2,4.054389
4,A1XNZ7PCE45KK7,B007I7Z3Z0,5,3.785923


In [ ]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
baseline_topn = build_topn_recommendations(
    lambda user_id, item_id: baseline_model.predict(user_id, item_id).est,
    top_n=TOP_N,
 )

bias_rank = ranking_metrics_from_topn(
    baseline_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

bias_precision = bias_rank["precision"]
bias_recall = bias_rank["recall"]
bias_map = bias_rank["map"]
bias_ndcg = bias_rank["ndcg"]

print("Baseline Bias Model Ranking Metrics")
print(f"Precision@{K}: {bias_precision:.8f}")
print(f"Recall@{K}: {bias_recall:.8f}")
print(f"MAP@{K}: {bias_map:.8f}")
print(f"NDCG@{K}: {bias_ndcg:.8f}")

Baseline Bias Model Ranking Metrics (full-candidate)
Precision@10: 0.0000
Recall@10: 0.0000
MAP@10: 0.0000
NDCG@10: 0.0000


#### 6.3.2 Catalog Metrics

In [18]:
if "baseline_topn" not in globals():
    baseline_topn = build_topn_recommendations(
        lambda user_id, item_id: baseline_model.predict(user_id, item_id).est,
        top_n=TOP_N,
    )

print("Baseline Bias full-candidate Top-N lists generated:", len(baseline_topn))

Baseline Bias full-candidate Top-N lists generated: 300


In [19]:
baseline_coverage = coverage_at_n(baseline_topn, all_items)
baseline_personalization = personalization_at_n(baseline_topn, all_items)
baseline_ils = intra_list_similarity_at_n(baseline_topn, item_similarity, item_to_idx)

print("Baseline Bias Behavioral/Catalog Metrics")
print(f"Coverage: {baseline_coverage:.4f}")
print(f"Personalization: {baseline_personalization:.4f}")
print(f"Intra-list similarity: {baseline_ils:.4f}")

Baseline Bias Behavioral/Catalog Metrics
Coverage: 0.0017
Personalization: 0.0126
Intra-list similarity: 0.3013


In [20]:
display(qualitative_examples(baseline_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,B000ED9L9E,4.769252
1,#oc-R115TNMSPFT9I7,2,B000EVG8FQ,4.752481
2,#oc-R115TNMSPFT9I7,3,B000EVIDWW,4.749454
3,#oc-R115TNMSPFT9I7,4,B001CWV4RS,4.738734
4,#oc-R115TNMSPFT9I7,5,B001CWSKFC,4.737346
5,#oc-R115TNMSPFT9I7,6,B001CWX7EG,4.736635
6,#oc-R115TNMSPFT9I7,7,B000EVG8HY,4.726577
7,#oc-R115TNMSPFT9I7,8,B001B4VOQI,4.725075
8,#oc-R115TNMSPFT9I7,9,B000255OIG,4.713257
9,#oc-R115TNMSPFT9I7,10,B000V17MLS,4.698530


### 6.4 Bayesian Average Popular Baseline

In [21]:
C = 20  # minimum reviews needed to trust a rating

item_stats = (
    train_df.groupby("ProductId")["Score"]
    .agg(["mean", "count"])
    .reset_index()
)

global_mean = train_df["Score"].mean()

item_stats["bap_score"] = (
    (item_stats["count"] * item_stats["mean"] + C * global_mean)
    / (item_stats["count"] + C)
)

bap_baseline = (
    item_stats
    .sort_values("bap_score", ascending=False)
    .rename(columns={"mean": "avg_rating", "count": "num_reviews"})
)[["ProductId", "avg_rating", "num_reviews", "bap_score"]]

bap_baseline.head(10)

,ProductId,avg_rating,num_reviews,bap_score
2072,B000EVIDWW,4.834951,103,4.730251
6362,B001B4VOQI,4.804511,133,4.724320
2062,B000EVG8FQ,4.825243,103,4.722121
6572,B001CWSKFC,4.819048,105,4.718567
6576,B001CWV4RS,4.820000,100,4.715174
6578,B001CWX7EG,4.809091,110,4.714007
423,B000255OIG,4.781022,137,4.705866
2066,B000EVG8HY,4.801887,106,4.704928
10517,B0030VJ79Q,4.809524,84,4.690586
10509,B0030VBPN2,4.807229,83,4.687582


#### 6.4.1 RMSE, MAE and Ranking Metrics

In [22]:
# Build a lookup: ProductId -> BAP score
bap_lookup = bap_baseline.set_index("ProductId")["bap_score"]

# For each test row, predict using BAP score (fallback to global mean if unseen)
global_mean = train_df["Score"].mean()
test_df["bap_pred"] = test_df["ProductId"].map(bap_lookup).fillna(global_mean)

# Keep the existing RMSE and MAE for BAP
bap_rmse = np.sqrt(((test_df["Score"] - test_df["bap_pred"]) ** 2).mean())
bap_mae = (test_df["Score"] - test_df["bap_pred"]).abs().mean()

bap_eval_df = test_df[["UserId", "ProductId", "Score", "bap_pred"]].rename(columns={"bap_pred": "pred"}).copy()

bap_eval_df.head()
print("Bayesian Average Popular")
print(f"RMSE: {bap_rmse:.4f}")
print(f"MAE:  {bap_mae:.4f}")

Bayesian Average Popular
RMSE: 1.2820
MAE:  1.0084


In [ ]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
bap_topn = build_topn_recommendations(
    lambda user_id, item_id: float(bap_lookup.get(item_id, global_mean)),
    top_n=TOP_N,
 )

bap_rank = ranking_metrics_from_topn(
    bap_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

bap_precision = bap_rank["precision"]
bap_recall = bap_rank["recall"]
bap_map = bap_rank["map"]
bap_ndcg = bap_rank["ndcg"]

print("Bayesian Average Popular Ranking Metrics")
print(f"Precision@{K}: {bap_precision:.8f}")
print(f"Recall@{K}: {bap_recall:.8f}")
print(f"MAP@{K}: {bap_map:.8f}")
print(f"NDCG@{K}: {bap_ndcg:.8f}")

Bayesian Average Popular Ranking Metrics (full-candidate)
Precision@10: 0.0000
Recall@10: 0.0000
MAP@10: 0.0000
NDCG@10: 0.0000


#### 6.4.2 Catalog Metrics

In [24]:
if "bap_topn" not in globals():
    bap_topn = build_topn_recommendations(
        lambda user_id, item_id: float(bap_lookup.get(item_id, global_mean)),
        top_n=TOP_N,
    )

print("Bayesian Average Popular full-candidate Top-N lists generated:", len(bap_topn))

Bayesian Average Popular full-candidate Top-N lists generated: 300


In [25]:
bap_coverage = coverage_at_n(bap_topn, all_items)
bap_personalization = personalization_at_n(bap_topn, all_items)
bap_ils = intra_list_similarity_at_n(bap_topn, item_similarity, item_to_idx)

print("Bayesian Average Popular Behavioral/Catalog Metrics")
print(f"Coverage: {bap_coverage:.4f}")
print(f"Personalization: {bap_personalization:.4f}")
print(f"Intra-list similarity: {bap_ils:.4f}")

Bayesian Average Popular Behavioral/Catalog Metrics
Coverage: 0.0007
Personalization: 0.0000
Intra-list similarity: 0.3233


In [26]:
display(qualitative_examples(bap_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,B000EVIDWW,4.730251
1,#oc-R115TNMSPFT9I7,2,B001B4VOQI,4.724320
2,#oc-R115TNMSPFT9I7,3,B000EVG8FQ,4.722121
3,#oc-R115TNMSPFT9I7,4,B001CWSKFC,4.718567
4,#oc-R115TNMSPFT9I7,5,B001CWV4RS,4.715174
5,#oc-R115TNMSPFT9I7,6,B001CWX7EG,4.714007
6,#oc-R115TNMSPFT9I7,7,B000255OIG,4.705866
7,#oc-R115TNMSPFT9I7,8,B000EVG8HY,4.704928
8,#oc-R115TNMSPFT9I7,9,B0030VJ79Q,4.690586
9,#oc-R115TNMSPFT9I7,10,B0030VBPN2,4.687582


This baseline treats items with more reviews as "their scores being more trustworthy", and pulls the ones with less reviews closer to the global mean, effectively treating those reviews as "less impactful". It is more reliable than ranking by average score alone.

### 6.5 Baseline comparison

In [27]:
baseline_results = pd.DataFrame({
    "Model": [
        "Random Baseline",
        "Baseline Bias Model",
        "Bayesian Average Popular"
    ],
    "RMSE": [
        random_rmse,
        baseline_rmse,
        bap_rmse
    ],
    "MAE": [
        random_mae,
        baseline_mae,
        bap_mae
    ],
    f"Precision@{K}": [
        random_precision,
        bias_precision,
        bap_precision
    ],
    f"Recall@{K}": [
        random_recall,
        bias_recall,
        bap_recall
    ],
    f"MAP@{K}": [
        random_map,
        bias_map,
        bap_map
    ],
    f"NDCG@{K}": [
        random_ndcg,
        bias_ndcg,
        bap_ndcg
    ],
    "Coverage": [
        random_coverage,
        baseline_coverage,
        bap_coverage
    ],
    "Personalization": [
        random_personalization,
        baseline_personalization,
        bap_personalization
    ],
    "Intra-list similarity": [
        random_ils,
        baseline_ils,
        bap_ils
    ]
})

display(baseline_results.sort_values("RMSE"))

,Model,RMSE,MAE,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage,Personalization,Intra-list similarity
1,Baseline Bias Model,1.221522,0.946435,0.0,0.0,0.0,0.0,0.001681,1.263545e-02,0.301281
2,Bayesian Average Popular,1.282010,1.008423,0.0,0.0,0.0,0.0,0.000700,1.110223e-16,0.323254
0,Random Baseline,1.630396,1.217879,0.0,0.0,0.0,0.0,0.004692,7.813200e-01,0.010171


The results already show a clear improvement from a random predictor to a simple bias based model. This shows that even basic patterns in the data have some predictive value. The bayesian average baseline also give us a strong non personalised reference which we can later compare against more personalized methods.

## **7. Collaborative Filtering**

### 7.1 User based collaborative filtering

User-based collaborative filtering predicts a user’s rating from the ratings of similar users.

In [28]:
sim_options_user = {
    "name": "pearson",
    "user_based": True
}

user_cf = KNNWithMeans(sim_options=sim_options_user)
user_cf.fit(trainset)

user_cf_predictions = user_cf.test(testset)

user_cf_rmse = accuracy.rmse(user_cf_predictions, verbose=False)
user_cf_mae = accuracy.mae(user_cf_predictions, verbose=False)

print("User-Based Collaborative Filtering")
print(f"RMSE: {user_cf_rmse:.4f}")
print(f"MAE:  {user_cf_mae:.4f}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
User-Based Collaborative Filtering
RMSE: 1.2092
MAE:  0.8859


#### 7.1.1 Ranking Metrics

In [29]:
usercf_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in user_cf_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

usercf_eval_df.head()

,UserId,ProductId,Score,pred
0,A1L01D2BD3RKVO,B000EVG8J2,5,4.428571
1,A3U62RE5XZDP0G,B0000BXJIS,5,4.191046
2,AOXC0JQQZGGB6,B008FHUFAU,3,4.191046
3,A3PWPNZVMNX3PA,B006BXV14E,2,4.191046
4,A1XNZ7PCE45KK7,B007I7Z3Z0,5,4.191046


In [ ]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
user_cf_topn = build_topn_recommendations(
    lambda user_id, item_id: user_cf.predict(user_id, item_id).est,
    top_n=TOP_N,
 )

user_cf_rank = ranking_metrics_from_topn(
    user_cf_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

user_cf_precision = user_cf_rank["precision"]
user_cf_recall = user_cf_rank["recall"]
user_cf_map = user_cf_rank["map"]
user_cf_ndcg = user_cf_rank["ndcg"]

print("User-Based CF Ranking Metrics")
print(f"Precision@{K}: {user_cf_precision:.4f}")
print(f"Recall@{K}: {user_cf_recall:.4f}")
print(f"MAP@{K}: {user_cf_map:.4f}")
print(f"NDCG@{K}: {user_cf_ndcg:.4f}")

User-Based CF Ranking Metrics (full-candidate)
Precision@10: 0.0016
Recall@10: 0.0090
MAP@10: 0.0040
NDCG@10: 0.0048


#### 7.1.1 Catalog Metrics

In [31]:
if "user_cf_topn" not in globals():
    user_cf_topn = build_topn_recommendations(
        lambda user_id, item_id: user_cf.predict(user_id, item_id).est,
        top_n=TOP_N,
    )

print("User-Based CF full-candidate Top-N lists generated:", len(user_cf_topn))

User-Based CF full-candidate Top-N lists generated: 300


In [32]:
user_cf_coverage = coverage_at_n(user_cf_topn, all_items)
user_cf_personalization = personalization_at_n(user_cf_topn, all_items)
user_cf_ils = intra_list_similarity_at_n(user_cf_topn, item_similarity, item_to_idx)

print("User-Based CF Behavioral/Catalog Metrics")
print(f"Coverage: {user_cf_coverage:.4f}")
print(f"Personalization: {user_cf_personalization:.4f}")
print(f"Intra-list similarity: {user_cf_ils:.4f}")

User-Based CF Behavioral/Catalog Metrics
Coverage: 0.0045
Personalization: 0.0370
Intra-list similarity: 0.0363


In [33]:
display(qualitative_examples(user_cf_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,0006641040,4.191046
1,#oc-R115TNMSPFT9I7,2,7310172001,4.191046
2,#oc-R115TNMSPFT9I7,3,7310172101,4.191046
3,#oc-R115TNMSPFT9I7,4,B00002N8SM,4.191046
4,#oc-R115TNMSPFT9I7,5,B00004CI84,4.191046
5,#oc-R115TNMSPFT9I7,6,B00004CXX9,4.191046
6,#oc-R115TNMSPFT9I7,7,B00004RAMV,4.191046
7,#oc-R115TNMSPFT9I7,8,B00004RAMX,4.191046
8,#oc-R115TNMSPFT9I7,9,B00004RAMY,4.191046
9,#oc-R115TNMSPFT9I7,10,B00004RBDU,4.191046


### 7.2 Item based collaborative filtering

Item-based collaborative filtering predicts preferences from relationships between products rather than between users.

In [34]:
sim_options_item = {
    "name": "pearson",
    "user_based": False
}

item_cf = KNNWithMeans(sim_options=sim_options_item)
item_cf.fit(trainset)

item_cf_predictions = item_cf.test(testset)

item_cf_rmse = accuracy.rmse(item_cf_predictions, verbose=False)
item_cf_mae = accuracy.mae(item_cf_predictions, verbose=False)

print("Item-Based Collaborative Filtering")
print(f"RMSE: {item_cf_rmse:.4f}")
print(f"MAE:  {item_cf_mae:.4f}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
Item-Based Collaborative Filtering
RMSE: 1.1949
MAE:  0.8596


#### 7.2.1 Ranking Metrics

In [35]:
itemcf_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in item_cf_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

itemcf_eval_df.head()

,UserId,ProductId,Score,pred
0,A1L01D2BD3RKVO,B000EVG8J2,5,4.983150
1,A3U62RE5XZDP0G,B0000BXJIS,5,4.191046
2,AOXC0JQQZGGB6,B008FHUFAU,3,4.191046
3,A3PWPNZVMNX3PA,B006BXV14E,2,4.191046
4,A1XNZ7PCE45KK7,B007I7Z3Z0,5,4.191046


In [ ]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
item_cf_topn = build_topn_recommendations(
    lambda user_id, item_id: item_cf.predict(user_id, item_id).est,
    top_n=TOP_N,
 )

item_cf_rank = ranking_metrics_from_topn(
    item_cf_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

item_cf_precision = item_cf_rank["precision"]
item_cf_recall = item_cf_rank["recall"]
item_cf_map = item_cf_rank["map"]
item_cf_ndcg = item_cf_rank["ndcg"]

print("Item-Based CF Ranking Metrics")
print(f"Precision@{K}: {item_cf_precision:.4f}")
print(f"Recall@{K}: {item_cf_recall:.4f}")
print(f"MAP@{K}: {item_cf_map:.4f}")
print(f"NDCG@{K}: {item_cf_ndcg:.4f}")

Item-Based CF Ranking Metrics (full-candidate)
Precision@10: 0.0005
Recall@10: 0.0054
MAP@10: 0.0009
NDCG@10: 0.0019


#### 7.2.2 Catalog Metrics

In [37]:
if "item_cf_topn" not in globals():
    item_cf_topn = build_topn_recommendations(
        lambda user_id, item_id: item_cf.predict(user_id, item_id).est,
        top_n=TOP_N,
    )

print("Item-Based CF full-candidate Top-N lists generated:", len(item_cf_topn))

Item-Based CF full-candidate Top-N lists generated: 300


In [38]:
item_cf_coverage = coverage_at_n(item_cf_topn, all_items)
item_cf_personalization = personalization_at_n(item_cf_topn, all_items)
item_cf_ils = intra_list_similarity_at_n(item_cf_topn, item_similarity, item_to_idx)

print("Item-Based CF Behavioral/Catalog Metrics")
print(f"Coverage: {item_cf_coverage:.4f}")
print(f"Personalization: {item_cf_personalization:.4f}")
print(f"Intra-list similarity: {item_cf_ils:.4f}")

Item-Based CF Behavioral/Catalog Metrics
Coverage: 0.0013
Personalization: 0.1530
Intra-list similarity: 0.0372


In [39]:
display(qualitative_examples(item_cf_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,0006641040,4.191046
1,#oc-R115TNMSPFT9I7,2,7310172001,4.191046
2,#oc-R115TNMSPFT9I7,3,7310172101,4.191046
3,#oc-R115TNMSPFT9I7,4,B00002N8SM,4.191046
4,#oc-R115TNMSPFT9I7,5,B00004CI84,4.191046
5,#oc-R115TNMSPFT9I7,6,B00004CXX9,4.191046
6,#oc-R115TNMSPFT9I7,7,B00004RAMV,4.191046
7,#oc-R115TNMSPFT9I7,8,B00004RAMX,4.191046
8,#oc-R115TNMSPFT9I7,9,B00004RAMY,4.191046
9,#oc-R115TNMSPFT9I7,10,B00004RBDU,4.191046


### 7.3 Model based collaborative filtering (SVD / Matrix factorization)

Model-based collaborative filtering learns latent user and item factors from the rating matrix. Here, we use SVD as the main model-based approach.

In [40]:
svd_model = SVD(random_state=42)
svd_model.fit(trainset)

svd_predictions = svd_model.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

print("SVD")
print(f"RMSE: {svd_rmse:.4f}")
print(f"MAE:  {svd_mae:.4f}")

SVD
RMSE: 1.1845
MAE:  0.8845


#### 7.3.1 Ranking Metrics

In [41]:
svd_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in svd_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

svd_eval_df.head()

,UserId,ProductId,Score,pred
0,A1L01D2BD3RKVO,B000EVG8J2,5,4.612524
1,A3U62RE5XZDP0G,B0000BXJIS,5,4.191046
2,AOXC0JQQZGGB6,B008FHUFAU,3,4.056734
3,A3PWPNZVMNX3PA,B006BXV14E,2,4.064782
4,A1XNZ7PCE45KK7,B007I7Z3Z0,5,3.783081


In [42]:
# Ranking metrics on full-candidate Top-N lists (unseen TRAIN catalog items)
svd_topn = build_topn_recommendations(
    lambda user_id, item_id: svd_model.predict(user_id, item_id).est,
    top_n=TOP_N,
 )

svd_rank = ranking_metrics_from_topn(
    svd_topn, eval_df=test_df, k=K, threshold=relevance_threshold
 )

svd_precision = svd_rank["precision"]
svd_recall = svd_rank["recall"]
svd_map = svd_rank["map"]
svd_ndcg = svd_rank["ndcg"]

print("SVD Ranking Metrics (full-candidate)")
print(f"Precision@{K}: {svd_precision:.4f}")
print(f"Recall@{K}: {svd_recall:.4f}")
print(f"MAP@{K}: {svd_map:.4f}")
print(f"NDCG@{K}: {svd_ndcg:.4f}")

SVD Ranking Metrics (full-candidate)
Precision@10: 0.0011
Recall@10: 0.0054
MAP@10: 0.0031
NDCG@10: 0.0037


#### 7.3.1 Ranking Metrics

In [43]:
if "svd_topn" not in globals():
    svd_topn = build_topn_recommendations(
        lambda user_id, item_id: svd_model.predict(user_id, item_id).est,
        top_n=TOP_N,
    )

print("SVD full-candidate Top-N lists generated:", len(svd_topn))

SVD full-candidate Top-N lists generated: 300


In [44]:
svd_coverage = coverage_at_n(svd_topn, all_items)
svd_personalization = personalization_at_n(svd_topn, all_items)
svd_ils = intra_list_similarity_at_n(svd_topn, item_similarity, item_to_idx)

print("SVD Behavioral/Catalog Metrics")
print(f"Coverage: {svd_coverage:.4f}")
print(f"Personalization: {svd_personalization:.4f}")
print(f"Intra-list similarity: {svd_ils:.4f}")

SVD Behavioral/Catalog Metrics
Coverage: 0.0111
Personalization: 0.1642
Intra-list similarity: 0.1089


In [45]:
display(qualitative_examples(svd_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,#oc-R115TNMSPFT9I7,1,B000ED9L9E,4.878714
1,#oc-R115TNMSPFT9I7,2,B000V17MLS,4.776939
2,#oc-R115TNMSPFT9I7,3,B001ONPMN2,4.771644
3,#oc-R115TNMSPFT9I7,4,B000EVG8FQ,4.763486
4,#oc-R115TNMSPFT9I7,5,B000EVIDWW,4.761811
5,#oc-R115TNMSPFT9I7,6,B001PICX42,4.760908
6,#oc-R115TNMSPFT9I7,7,B0012KB47K,4.757433
7,#oc-R115TNMSPFT9I7,8,B00283OXTG,4.751459
8,#oc-R115TNMSPFT9I7,9,B001CWV4RS,4.742907
9,#oc-R115TNMSPFT9I7,10,B0012KH0F0,4.741467


### 7.4 Collaborative filtering comparison

In [46]:
cf_results = pd.DataFrame({
    "Model": [
        "User-Based CF",
        "Item-Based CF",
        "SVD"
    ],
    "RMSE": [
        user_cf_rmse,
        item_cf_rmse,
        svd_rmse
    ],
    "MAE": [
        user_cf_mae,
        item_cf_mae,
        svd_mae
    ],
    f"Precision@{K}": [
        user_cf_precision,
        item_cf_precision,
        svd_precision
    ],
    f"Recall@{K}": [
        user_cf_recall,
        item_cf_recall,
        svd_recall
    ],
    f"MAP@{K}": [
        user_cf_map,
        item_cf_map,
        svd_map
    ],
    f"NDCG@{K}": [
        user_cf_ndcg,
        item_cf_ndcg,
        svd_ndcg
    ],
    "Coverage": [
        user_cf_coverage,
        item_cf_coverage,
        svd_coverage
    ],
    "Personalization": [
        user_cf_personalization,
        item_cf_personalization,
        svd_personalization
    ],
    "Intra-list similarity": [
        user_cf_ils,
        item_cf_ils,
        svd_ils
    ]
})

cf_results.sort_values("RMSE")

,Model,RMSE,MAE,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage,Personalization,Intra-list similarity
2,SVD,1.184499,0.884478,0.001075,0.005376,0.003136,0.003728,0.011135,0.164183,0.108930
1,Item-Based CF,1.194936,0.859609,0.000538,0.005376,0.000896,0.001915,0.001331,0.153041,0.037151
0,User-Based CF,1.209167,0.885879,0.001613,0.008961,0.004032,0.004768,0.004482,0.037039,0.036287


The results show a clear contrast between prediction accuracy, ranking performance, and actual recommendation quality, and this contrast is both expected and informative.

First, all three models appear almost identical in terms of ranking metrics, with Precision@10 around 0.787, Recall@10 close to 1.0, and NDCG@10 above 0.97. At first glance, this suggests that all models perform extremely well and almost equally in ranking relevant items. However, this is largely a consequence of the evaluation setup: since ranking metrics were computed using test-only interactions, the models are effectively ranking a small set of already relevant items against each other rather than competing against the full catalog. This makes the task artificially easy and leads to inflated and very similar scores across models. Therefore, the lack of differentiation in ranking metrics is not surprising and does not necessarily reflect real-world recommendation performance.

Looking at regression metrics, Item-Based CF achieves the best RMSE and MAE, indicating that it is the most accurate at predicting rating values. This is typical for memory-based collaborative filtering methods, which rely heavily on observed similarities and can fit known interactions closely. However, this strength does not translate into better recommendation behavior. In fact, Item-Based CF performs extremely poorly in the behavioral metrics: its coverage is almost zero (0.0012), and personalization is also near zero (0.0119). This means that the model is recommending almost the same very small subset of items to all users. This is a classic symptom of popularity bias, where the model over-relies on frequently interacted items and ignores most of the catalog. This behavior is entirely expected for item-based approaches, especially in sparse datasets with long-tail distributions.

User-Based CF improves upon this by increasing both coverage (0.0182) and personalization (0.58). This indicates that recommendations start to differ across users and include a broader set of items. The reason is that user-based methods consider neighborhoods of similar users, which introduces more variability than item-based methods that tend to converge on popular items. However, the improvement is still limited, and the model does not fully overcome the bias toward popular items.

SVD presents the most balanced and arguably the best performance overall. Although its RMSE is slightly worse than Item-Based CF, it significantly outperforms both memory-based methods in behavioral metrics. Coverage increases to 0.0303, meaning a much larger portion of the catalog is being recommended. Personalization is very high (0.893), indicating that users receive highly distinct recommendation lists. Intra-list similarity is also higher, suggesting that recommended items are more coherent within each list. This behavior is expected because SVD, as a latent factor model, captures underlying user preferences and item characteristics beyond direct co-occurrence patterns. This allows it to generalize better and recommend less obvious, more personalized items.

One potentially surprising result is how similar the ranking metrics are across all models despite large differences in behavioral performance. However, given that ranking was evaluated only on test interactions, this is not unexpected. The evaluation setup masks the true differences in recommendation quality by not considering the full candidate space. If ranking metrics were computed using full-candidate evaluation, larger differences between models would likely emerge, aligning more closely with the behavioral metrics.

In summary, the results are consistent with theoretical expectations. Memory-based methods (especially item-based) tend to optimize prediction accuracy but suffer from popularity bias and lack of personalization. Latent factor models like SVD sacrifice some prediction accuracy but produce significantly better recommendation behavior. The main limitation in the analysis comes from the test-only ranking evaluation, which overestimates ranking performance and reduces its ability to distinguish between models.


##############################################


## 8. Artifact Export for Hybrid

Export CF and baseline score artifacts for the hybrid notebook.

In [47]:

project_dir = Path.cwd()
baselines_dir = project_dir / "artifacts" / "baselines_cf"
baselines_dir.mkdir(parents=True, exist_ok=True)

# Best CF model for hybrid (SVD selected from this notebook's CF section).
cf_test_scores = svd_eval_df.rename(columns={"pred": "cf_score"})[["UserId", "ProductId", "cf_score"]].copy()

cf_topn_scores = (
    pd.DataFrame(
        [(u, i, s) for u, recs in svd_topn.items() for i, s in recs],
        columns=["UserId", "ProductId", "cf_score"]
    )
    if len(svd_topn) > 0
    else pd.DataFrame(columns=["UserId", "ProductId", "cf_score"])
)

# Popularity/baseline fallback for hybrid.
pop_scores = bap_baseline[["ProductId", "bap_score"]].rename(columns={"bap_score": "pop_score"}).copy()

cf_test_scores.to_csv(baselines_dir / "cf_test_scores.csv", index=False)
cf_topn_scores.to_csv(baselines_dir / "cf_topn_scores.csv", index=False)
pop_scores.to_csv(baselines_dir / "pop_scores.csv", index=False)

baseline_results.to_csv(baselines_dir / "baseline_results.csv", index=False)
cf_results.to_csv(baselines_dir / "cf_results.csv", index=False)

artifact_manifest = pd.DataFrame(
    [
        {"artifact": "cf_test_scores.csv", "rows": len(cf_test_scores)},
        {"artifact": "cf_topn_scores.csv", "rows": len(cf_topn_scores)},
        {"artifact": "pop_scores.csv", "rows": len(pop_scores)},
        {"artifact": "baseline_results.csv", "rows": len(baseline_results)},
        {"artifact": "cf_results.csv", "rows": len(cf_results)},
    ]
)
artifact_manifest.to_csv(baselines_dir / "artifact_manifest.csv", index=False)

print("Saved baselines/CF artifacts to:", baselines_dir)
display(artifact_manifest)

Saved baselines/CF artifacts to: /Users/onosaeka/Uni/BCSAI/3-2/Chatbot&RecSys/Recommender-Sys/artifacts/baselines_cf


,artifact,rows
0,cf_test_scores.csv,113691
1,cf_topn_scores.csv,3000
2,pop_scores.csv,14279
3,baseline_results.csv,3
4,cf_results.csv,3
